# Name That LLM — offline inference (internet disabled)

Loads Qwen3-8B-Base in 4-bit NF4 from the attached Kaggle model, applies the trained
LoRA adapter + `score` head from the attached `engs108-qwen3-8b-lora-adapter` dataset,
and writes `submission.parquet`. `bitsandbytes` is installed from a wheel in the attached
`engs108-offline-wheels` dataset; nothing is downloaded.

In [ ]:
import glob
import subprocess
import sys

wheels = sorted(glob.glob("/kaggle/input/**/bitsandbytes-*.whl", recursive=True))
assert wheels, "bitsandbytes wheel not found; attach the engs108-offline-wheels dataset"
print("Installing", wheels[-1])
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps", wheels[-1]],
    check=True,
)

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import time

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import torch

RUN_START = time.time()

DATA_DIR = "/kaggle/input/competitions/name-that-llm"
MODEL_PATH = (
    "/kaggle/input/models/qwen-lm/qwen-3/"
    "transformers/8b-base/1"
)
OUTPUT_DIR = "/kaggle/working"
MAX_LENGTH = 512
BATCH_SIZE = 8
SEED = 42
VAL_SIZE = 0.15
RUN_VAL_CHECK = True  # re-score the training run's held-out prompt groups as a sanity check

adapter_configs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
assert len(adapter_configs) == 1, adapter_configs
ADAPTER_DIR = os.path.dirname(adapter_configs[0])
print("Adapter dir:", ADAPTER_DIR, sorted(os.listdir(ADAPTER_DIR)))

print("CUDA available:", torch.cuda.is_available(), "| visible GPUs:", torch.cuda.device_count())

In [ ]:
import importlib.metadata as metadata

for package in ["torch", "transformers", "peft", "accelerate", "bitsandbytes", "scikit-learn"]:
    print(package, metadata.version(package))

In [ ]:
from peft import PeftModel
from transformers import AutoModelForSequenceClassification, AutoTokenizer, BitsAndBytesConfig

train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("Test:", test.shape)

with open(f"{ADAPTER_DIR}/label_map.json") as f:
    id2label = {int(k): v for k, v in json.load(f)["id2label"].items()}
label2id = {v: k for k, v in id2label.items()}
model_names = [id2label[i] for i in range(len(id2label))]
assert model_names == sorted(train["model"].unique()), "label map does not match train labels"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=18,
    id2label=id2label,
    label2id=label2id,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    local_files_only=True,
)
base.config.pad_token_id = tokenizer.pad_token_id

model = PeftModel.from_pretrained(base, ADAPTER_DIR, is_trainable=False)
model.eval()
print("Loaded adapter; GPU memory:", f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
@torch.no_grad()
def predict_labels(texts, batch_size=BATCH_SIZE):
    order = np.argsort([len(t) for t in texts])[::-1]
    pred_ids = np.empty(len(texts), dtype=int)
    for start in range(0, len(texts), batch_size):
        idx = order[start:start + batch_size]
        enc = tokenizer([texts[i] for i in idx], truncation=True, max_length=MAX_LENGTH,
                        padding=True, return_tensors="pt").to(model.device)
        with torch.autocast("cuda", dtype=torch.float16):
            logits = model(**enc).logits
        pred_ids[idx] = logits.argmax(-1).cpu().numpy()
    return [id2label[int(i)] for i in pred_ids]

In [ ]:
if RUN_VAL_CHECK:
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import GroupShuffleSplit

    splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=SEED)
    _, val_idx = next(splitter.split(train, groups=train["prompt_id"]))
    val_df = train.iloc[val_idx]
    t0 = time.time()
    val_preds = predict_labels(val_df["text"].tolist())
    print(f"Held-out val ({len(val_df)} rows, {val_df['prompt_id'].nunique()} prompts) "
          f"balanced accuracy: {balanced_accuracy_score(val_df['model'], val_preds):.4f} "
          f"[{time.time() - t0:.0f}s]")

In [ ]:
t0 = time.time()
test_preds = predict_labels(test["text"].tolist())
print(f"Predicted {len(test)} test rows in {time.time() - t0:.1f}s")

submission = pd.DataFrame({"id": test["id"], "model": test_preds})
sample_schema = pq.read_schema(f"{DATA_DIR}/sample_submission.parquet")
table = pa.Table.from_pandas(submission, preserve_index=False).cast(sample_schema)
pq.write_table(table, f"{OUTPUT_DIR}/submission.parquet")

written = pq.read_table(f"{OUTPUT_DIR}/submission.parquet")
assert written.schema.equals(sample_schema, check_metadata=False), (written.schema, sample_schema)
assert written.num_rows == len(test)
assert written.column("id").to_pylist() == test["id"].tolist()
assert set(written.column("model").to_pylist()) <= set(model_names)
print(written.schema)
print(written.to_pandas().head(10))
print(f"Total runtime: {(time.time() - RUN_START) / 60:.1f} min")

In [ ]:
# Confirms the session really had no internet access.
import urllib.request

try:
    urllib.request.urlopen("https://pypi.org", timeout=5)
    print("Internet: REACHABLE")
except Exception as e:
    print("Internet: unreachable ->", type(e).__name__)